 Batch Normalization & Layer Normalization

In [1]:
import torch
import numpy as np

# Disable Autograd
torch.set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

 Input Data

In [2]:
torch.manual_seed(42)

X = torch.randn(4,5)

gamma = torch.ones(5)
beta = torch.zeros(5)

eps = 1e-5

Batch Normalization

In [3]:
print("========== Batch Normalization ==========\n")

# Forward Pass

mean = torch.mean(X, dim=0)
variance = torch.var(X, dim=0, unbiased=False)

X_hat = (X - mean) / torch.sqrt(variance + eps)

Y = gamma * X_hat + beta

print("Input")
print(X)

print("\nMean")
print(mean)

print("\nVariance")
print(variance)

print("\nNormalized Output")
print(Y)

# Running Mean & Variance

running_mean = mean
running_var = variance

print("\nRunning Mean")
print(running_mean)

print("\nRunning Variance")
print(running_var)

# ==========================================
# Backward Pass
# ==========================================

print("\n========== BatchNorm Backward ==========\n")

dY = torch.ones_like(Y)

dgamma = torch.sum(dY * X_hat, dim=0)

dbeta = torch.sum(dY, dim=0)

dXhat = dY * gamma

std_inv = 1 / torch.sqrt(variance + eps)

N = X.shape[0]

dvar = torch.sum(
    dXhat * (X - mean) * -0.5 * std_inv**3,
    dim=0
)

dmean = torch.sum(
    -dXhat * std_inv,
    dim=0
) + dvar * torch.mean(
    -2 * (X - mean),
    dim=0
)

dX = (
    dXhat * std_inv
    + dvar * 2 * (X - mean) / N
    + dmean / N
)

print("Gradient Gamma")
print(dgamma)

print("\nGradient Beta")
print(dbeta)

print("\nGradient Input")
print(dX)


========== Batch Normalization ==========

Input
tensor([[ 1.9269,  1.4873,  0.9007, -2.1055, -0.7581],
        [ 1.0783,  0.8008,  1.6806,  0.3559, -0.6866],
        [-0.4934,  0.2415, -0.2316,  0.0418, -0.2516],
        [ 0.8599, -0.3097, -0.3957,  0.8034, -0.6216]])

Mean
tensor([ 0.8429,  0.5550,  0.4885, -0.2261, -0.5795])

Variance
tensor([0.7541, 0.4439, 0.7229, 1.2506, 0.0382])

Normalized Output
tensor([[ 1.2483,  1.3993,  0.4848, -1.6806, -0.9143],
        [ 0.2711,  0.3690,  1.4021,  0.5204, -0.5483],
        [-1.5388, -0.4705, -0.8470,  0.2395,  1.6781],
        [ 0.0195, -1.2978, -1.0400,  0.9206, -0.2155]])

Running Mean
tensor([ 0.8429,  0.5550,  0.4885, -0.2261, -0.5795])

Running Variance
tensor([0.7541, 0.4439, 0.7229, 1.2506, 0.0382])

========== BatchNorm Backward ==========

Gradient Gamma
tensor([ 9.8720e-08,  1.1921e-07,  2.3842e-07,  5.9605e-08, -2.5332e-07])

Gradient Beta
tensor([4., 4., 4., 4., 4.])

Gradient Input
tensor([[ 0.0000e+00, -1.1921e-07,  0.0000e+

 Layer Normalization

In [5]:
print("\n========== Layer Normalization ==========\n")

mean_ln = torch.mean(X, dim=1, keepdim=True)

var_ln = torch.var(X, dim=1, unbiased=False, keepdim=True)

X_hat_ln = (X - mean_ln) / torch.sqrt(var_ln + eps)

Y_ln = gamma * X_hat_ln + beta

print("LayerNorm Output")
print(Y_ln)


========== Layer Normalization ==========

LayerNorm Output
tensor([[ 1.0876,  0.7954,  0.4057, -1.5920, -0.6967],
        [ 0.5457,  0.1956,  1.3055, -0.3658, -1.6810],
        [-1.3927,  1.4926, -0.3650,  0.7084, -0.4434],
        [ 1.2529, -0.5959, -0.7318,  1.1636, -1.0888]])


LayerNorm Backward

In [6]:
print("\n========== LayerNorm Backward ==========\n")

dY_ln = torch.ones_like(Y_ln)

dgamma_ln = torch.sum(dY_ln * X_hat_ln, dim=0)

dbeta_ln = torch.sum(dY_ln, dim=0)

dXhat = dY_ln * gamma

std_inv = 1 / torch.sqrt(var_ln + eps)

D = X.shape[1]

dvar = torch.sum(
    dXhat * (X - mean_ln) * -0.5 * std_inv**3,
    dim=1,
    keepdim=True
)

dmean = torch.sum(
    -dXhat * std_inv,
    dim=1,
    keepdim=True
) + dvar * torch.mean(
    -2 * (X - mean_ln),
    dim=1,
    keepdim=True
)

dX_ln = (
    dXhat * std_inv
    + dvar * 2 * (X - mean_ln) / D
    + dmean / D
)

print("Gradient Gamma")
print(dgamma_ln)

print("\nGradient Beta")
print(dbeta_ln)

print("\nGradient Input")
print(dX_ln)


========== LayerNorm Backward ==========

Gradient Gamma
tensor([ 1.4934,  1.8878,  0.6144, -0.0857, -3.9098])

Gradient Beta
tensor([4., 4., 4., 4., 4.])

Gradient Input
tensor([[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  1.1921e-07,  0.0000e+00, -1.1921e-07],
        [-2.3842e-07,  2.3842e-07,  0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00]])
